# 01 · Explore SwimXYZ labels

**Goal:** find out exactly what is inside the SwimXYZ label archives before writing any parser.

- Downloads **one** label zip (Backstroke, ~1.6 GB) from Zenodo, never the 300 GB of video.
- Lists the archive structure, then peeks inside one sample of each file type.
- Answers the open questions: per-frame or per-video files? 2D coords in image pixels? Which camera view per file? Is global (pool) position stored, so speed can be computed?

Runtime: CPU is fine for this notebook.

In [ ]:
# Settings
STROKE = "Backstroke"            # Backstroke | Breaststroke | Butterfly | Freestyle
ZENODO_RECORD = "8399376"        # labels + SMPL motions record
ZIP_NAME = f"{STROKE}_labels.zip"
URL = f"https://zenodo.org/records/{ZENODO_RECORD}/files/{ZIP_NAME}?download=1"

USE_DRIVE = True                 # cache the zip on Google Drive so later sessions skip the download
LOCAL_DIR = "/content/swimxyz"
DRIVE_DIR = "/content/drive/MyDrive/swimxyz"

In [ ]:
import os, shutil, subprocess

os.makedirs(LOCAL_DIR, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/raw", exist_ok=True)

local_zip = f"{LOCAL_DIR}/{ZIP_NAME}"
drive_zip = f"{DRIVE_DIR}/raw/{ZIP_NAME}"

if os.path.exists(local_zip):
    print("already on local disk")
elif USE_DRIVE and os.path.exists(drive_zip):
    print("copying from Drive cache...")
    shutil.copy(drive_zip, local_zip)
else:
    print("downloading from Zenodo...")
    subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", local_zip, URL], check=True)
    if USE_DRIVE:
        shutil.copy(local_zip, drive_zip)
print(f"{local_zip}: {os.path.getsize(local_zip)/1e9:.2f} GB")

In [ ]:
# Archive structure without extracting
import zipfile
from collections import Counter

zf = zipfile.ZipFile(local_zip)
infos = [i for i in zf.infolist() if not i.is_dir()]
print(f"{len(infos):,} files, {sum(i.file_size for i in infos)/1e9:.2f} GB uncompressed\n")

ext = Counter(os.path.splitext(i.filename)[1].lower() or "<none>" for i in infos)
print("extensions:", dict(ext.most_common()))

def depth_counter(d):
    return Counter("/".join(i.filename.split("/")[:d]) for i in infos)

for d in (1, 2, 3):
    c = depth_counter(d)
    print(f"\n--- depth {d}: {len(c)} distinct prefixes (first 15) ---")
    for k, v in list(c.items())[:15]:
        print(f"{v:>8}  {k}")

print("\n--- first 40 paths ---")
for i in infos[:40]:
    print(f"{i.file_size:>10}  {i.filename}")

In [ ]:
# Extract to fast local disk
EXTRACT_DIR = f"{LOCAL_DIR}/{STROKE}_labels"
if not os.path.exists(EXTRACT_DIR):
    zf.extractall(EXTRACT_DIR)
print("extracted to", EXTRACT_DIR)

In [ ]:
# Generic peek: prints structure of any common file type
import json, pickle
import numpy as np

def describe(obj, indent=0, max_items=8):
    pad = "  " * indent
    if isinstance(obj, dict):
        print(f"{pad}dict[{len(obj)}]")
        for k in list(obj)[:max_items]:
            print(f"{pad}  [{k!r}]:", end=" ")
            describe(obj[k], indent + 2, max_items)
        if len(obj) > max_items: print(f"{pad}  ... {len(obj)-max_items} more keys")
    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{type(obj).__name__}[{len(obj)}]")
        if obj:
            print(f"{pad}  [0]:", end=" ")
            describe(obj[0], indent + 2, max_items)
    elif isinstance(obj, np.ndarray):
        print(f"ndarray {obj.shape} {obj.dtype}  min={np.nanmin(obj) if obj.size else None} max={np.nanmax(obj) if obj.size else None}")
    else:
        s = repr(obj)
        print(s[:120] + ("..." if len(s) > 120 else ""))

def peek(path, n_lines=15):
    print("=" * 100); print(path, f"({os.path.getsize(path):,} bytes)"); print("=" * 100)
    e = os.path.splitext(path)[1].lower()
    try:
        if e == ".json":
            with open(path) as f: describe(json.load(f))
        elif e in (".npy",):
            describe(np.load(path))
        elif e == ".npz":
            d = np.load(path); describe({k: d[k] for k in d.files})
        elif e in (".pkl", ".pickle"):
            with open(path, "rb") as f: describe(pickle.load(f))
        else:  # csv / txt / unknown: show raw lines
            with open(path, errors="replace") as f:
                for _ in range(n_lines):
                    line = f.readline()
                    if not line: break
                    print(line.rstrip()[:200])
    except Exception as ex:
        print("could not parse:", ex)

In [ ]:
# Folder tree: distinct values at each level, and number of clip folders
# Expected layout: Stroke/View/Swimmer/Water/Lighting/Speed/Position/<Format>/<file>.txt
from collections import defaultdict
levels = defaultdict(Counter)
clip_dirs = set()
file_names = Counter()
for root, _, files in os.walk(EXTRACT_DIR):
    rel = os.path.relpath(root, EXTRACT_DIR).split(os.sep)
    for fn in files:
        file_names["/".join(rel[7:] + [fn])] += 1   # path below the Position level
        clip_dirs.add("/".join(rel[:7]))
for d in clip_dirs:
    for lvl, part in enumerate(d.split("/")):
        levels[lvl][part] += 1
print(f"{len(clip_dirs)} clip folders\n")
for lvl in sorted(levels):
    print(f"level {lvl}: {dict(levels[lvl])}")
print("\nfiles inside each clip folder (name: count):")
for k, v in sorted(file_names.items()):
    print(f"  {v:>6}  {k}")

In [ ]:
# Parse every file of ONE clip properly (semicolon-separated, decimal comma)
import pandas as pd

def load_txt(path):
    return pd.read_csv(path, sep=";", decimal=",")

clip = sorted(clip_dirs)[0]
clip_path = os.path.join(EXTRACT_DIR, clip)
print("clip:", clip, "\n")
for root, _, files in os.walk(clip_path):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        df = load_txt(p)
        df = df.loc[:, ~df.columns.str.startswith("Unnamed")]   # trailing ';' makes an empty column
        print("=" * 90)
        print(os.path.relpath(p, clip_path), "| shape", df.shape)
        cols = list(df.columns)
        print("columns:", cols[:9], "...", cols[-6:])
        axes = sorted({c.split(".")[-1] for c in cols if "." in c})
        for a in axes:
            v = df[[c for c in cols if c.endswith("." + a)]].to_numpy()
            print(f"  .{a}: min {v.min():9.3f}  max {v.max():9.3f}")

In [ ]:
# Speed check: does any 3D file hold a GLOBAL track? Pelvis = mean of hips.
for root, _, files in os.walk(clip_path):
    for fn in sorted(files):
        if not fn.lower().startswith("3d"):
            continue
        df = load_txt(os.path.join(root, fn))
        hips = [c for c in df.columns if c.split(".")[0] in ("RHip", "LHip")]
        if not hips:
            continue
        P = np.stack([df[[f"{j}.{a}" for j in ("RHip", "LHip")]].mean(axis=1) for a in "xyz"], axis=1)
        disp = P[-1] - P[0]
        dur = (len(df) - 1) / 60
        print(f"{os.path.relpath(os.path.join(root, fn), clip_path):40s} frames={len(df)} "
              f"displacement(x,y,z)={np.round(disp, 2)} -> {np.linalg.norm(disp)/dur:.2f} units/s")

In [ ]:
# Is the motion identical across the 5 camera views? (decides how to split train/test)
import hashlib
groups = defaultdict(list)
for d in clip_dirs:
    parts = d.split("/")
    key = "/".join(parts[:1] + parts[2:])       # everything except the View level
    f = os.path.join(EXTRACT_DIR, d, "COCO", "3D_pelvis.txt")
    if os.path.exists(f):
        groups[key].append(hashlib.md5(open(f, "rb").read()).hexdigest())
same = sum(len(set(v)) == 1 for v in groups.values())
print(f"{len(groups)} motion groups; {same} have byte-identical 3D_pelvis across views")
print("views per group:", Counter(len(v) for v in groups.values()))

In [ ]:
# Plot one clip's 2D track to see image coordinates
import matplotlib.pyplot as plt
two_d = [os.path.join(r, f) for r, _, fs in os.walk(clip_path) for f in fs if f.lower().startswith("2d")]
print(two_d)
if two_d:
    df = load_txt(two_d[0])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for j in ("Nose", "RWrist", "LWrist", "RAnkle"):
        if f"{j}.x" in df:
            ax[0].plot(df[f"{j}.x"], df[f"{j}.y"], label=j, lw=1)
            ax[1].plot(df[f"{j}.y"], label=j, lw=1)
    ax[0].invert_yaxis(); ax[0].set_title("2D trajectory"); ax[0].legend()
    ax[1].set_title("y over frames"); ax[1].set_xlabel("frame")
    plt.show()

## Part 2 · Checks before writing the parser

Findings so far: `COCO/` files are NaN, `body25/` and `base/` are filled. `*_cam` = camera frame (2D in pixels, 3D in metres),
`*_pelvis` = pelvis-centred. The 2D_pelvis plot flips sides at frame ~65.

These cells answer:
1. How much is NaN in each format?
2. Is the flip only in `2D_pelvis` (then we normalise `2D_cam` ourselves)?
3. How many **truly different motions** exist? (decides the train/test split)
4. Is `3D_cam` speed consistent across the 5 views? (= cameras are static, so speed ground truth is valid)

In [ ]:
# Helpers
def load_joints(clip, fmt="body25", kind="3D_pelvis"):
    df = load_txt(os.path.join(EXTRACT_DIR, clip, fmt, f"{kind}.txt"))
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    names = list(dict.fromkeys(c.rsplit(".", 1)[0] for c in df.columns))
    return df.to_numpy(np.float32).reshape(len(df), len(names), 3), names

def meta(clip):
    p = clip.split("/")
    return dict(stroke=p[0], view=p[1], swimmer=p[2], water=p[3], light=p[4],
                speed=p[5], pos=p[6], group="/".join([p[0]] + p[2:]))

clips = sorted(clip_dirs)
import random
random.seed(0)
sample = random.sample(clips, 100)

In [ ]:
# 1. NaN per format
for fmt in ("COCO", "body25", "base"):
    for kind in ("2D_cam", "3D_cam"):
        fr, nj = [], Counter()
        for c in sample:
            a, names = load_joints(c, fmt, kind)
            m = np.isnan(a).any(axis=2)
            fr.append(m.mean())
            for j in np.where(m.any(0))[0]:
                nj[names[j]] += 1
        print(f"{fmt:7s} {kind:7s} NaN fraction {np.mean(fr):.3f} | joints with NaN (in n clips): {dict(nj.most_common(6))}")

In [ ]:
# 2. Sudden jumps: 2D_cam vs 2D_pelvis (largest frame-to-frame move of any joint, pixels)
def max_jump(a):
    return np.nanmax(np.linalg.norm(np.diff(a[:, :, :2], axis=0), axis=2))

rows = []
for c in sample:
    for kind in ("2D_cam", "2D_pelvis"):
        rows.append(dict(view=meta(c)["view"], kind=kind, jump=max_jump(load_joints(c, "body25", kind)[0])))
jdf = pd.DataFrame(rows)
print(jdf.groupby(["kind", "view"]).jump.describe()[["50%", "max"]].round(1))
print("\nclips with a >100 px jump:", jdf[jdf.jump > 100].groupby("kind").size().to_dict())

a, names = load_joints(clip, "body25", "2D_cam")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for j in ("Nose", "RWrist", "LWrist", "RAnkle", "MidHip"):
    i = names.index(j)
    ax[0].plot(a[:, i, 0], a[:, i, 1], label=j, lw=1)
    ax[1].plot(a[:, i, 1], label=j, lw=1)
ax[0].invert_yaxis(); ax[0].legend(); ax[0].set_title("body25 2D_cam (pixels)"); ax[1].set_title("y over frames")
plt.show()

In [ ]:
# 3. How many truly different motions? Compare clips with a rotation/reflection-invariant signature:
#    distances between joint pairs over time, from body25/3D_pelvis. Loads all clips (~1-2 min).
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

P3 = {c: load_joints(c, "body25", "3D_pelvis")[0] for c in clips}
names = load_joints(clips[0], "body25", "3D_pelvis")[1]
idx = {n: i for i, n in enumerate(names)}
pairs = [("RWrist", "LWrist"), ("RAnkle", "LAnkle"), ("Nose", "RAnkle"), ("RWrist", "LAnkle"), ("LWrist", "RKnee")]
T = min(len(a) for a in P3.values())
S = np.stack([np.concatenate([np.linalg.norm(P3[c][:T, idx[i]] - P3[c][:T, idx[j]], axis=1) for i, j in pairs])
              for c in clips])
S = np.nan_to_num(S)
D = cdist(S, S, "cityblock") / S.shape[1]        # mean absolute difference in metres

nn = np.sort(D, axis=1)[:, 1]
print("nearest other clip, mean abs diff (m): quantiles 0/10/50/90/100% =", np.quantile(nn, [0, .1, .5, .9, 1]).round(4))
for th in (0.005, 0.01, 0.02, 0.05):
    n_comp, lab = connected_components(csr_matrix(D < th), directed=False)
    comp_groups = defaultdict(set); group_comps = defaultdict(set)
    for c, l in zip(clips, lab):
        comp_groups[l].add(meta(c)["group"]); group_comps[meta(c)["group"]].add(l)
    print(f"threshold {th}: {n_comp} distinct motions | motions shared by >1 parameter group: "
          f"{sum(len(g) > 1 for g in comp_groups.values())} | groups whose 5 views differ: "
          f"{sum(len(s) > 1 for s in group_comps.values())}")

In [ ]:
# 4. Speed from 3D_cam (MidHip, metres). If cameras are static, all 5 views of one group agree.
rows = []
for c in clips:
    a, names = load_joints(c, "body25", "3D_cam")
    p = a[:, names.index("MidHip")]
    ok = np.where(~np.isnan(p).any(1))[0]
    dur = (ok[-1] - ok[0]) / 60
    rows.append(dict(**meta(c), clip=c,
                     speed_ms=np.linalg.norm(p[ok[-1]] - p[ok[0]]) / dur,
                     path_ms=np.nansum(np.linalg.norm(np.diff(p[ok], axis=0), axis=1)) / dur))
sp = pd.DataFrame(rows)
print(sp.groupby(["speed", "view"]).speed_ms.describe()[["mean", "std", "min", "max"]].round(3))
print("\nspread of speed across the 5 views of one group (m/s):")
print(sp.groupby("group").speed_ms.agg(lambda s: s.max() - s.min()).describe().round(3))

## Part 3 · Re-run with the broken last frame removed

Part 2 output showed every joint jumping at the **last frame (row 301)** in `2D_cam` (y-over-frames plot), which
contaminated check 2 (max jump) and check 4 (speed uses first vs last frame). These cells drop that frame and answer:

- 2b. After that, where and how big are the remaining jumps? (Front view: is it a near-camera projection blow-up?)
- 3b. Do the 2 "distinct motions" from check 3 equal the `Speed_2` / `Speed_3` setting?
- 4b. Robust speed (line fit to MidHip, not endpoints): do the 5 views of a group agree now?
- 5. What is in `smpl_swimming_motions.zip` (the paper's 240 motions = 60 per stroke)?

In [ ]:
# 2b. Jumps without the last frame
def load_clean(clip, kind):
    a, names = load_joints(clip, "body25", kind)
    return a[:-2], names                       # rows 300-301 are reset frames

rows = []
for c in sample:
    for kind in ("2D_cam", "2D_pelvis", "3D_cam"):
        a, names = load_clean(c, kind)
        xy = a if kind == "3D_cam" else a[:, :, :2]
        d = np.linalg.norm(np.diff(xy, axis=0), axis=2)          # (T-1, J) per-frame move
        f, j = np.unravel_index(np.nanargmax(d), d.shape)
        rows.append(dict(view=meta(c)["view"], kind=kind, jump=d[f, j], typical=np.nanmedian(d),
                         frame=f, joint=names[j], min_depth=a[:, :, 2].min() if kind == "2D_cam" else np.nan))
jdf = pd.DataFrame(rows)
jdf["ratio"] = jdf.jump / jdf.typical
print(jdf.groupby(["kind", "view"])[["typical", "jump", "ratio", "min_depth"]].median().round(2))
print("\nframe of the biggest jump:\n", jdf.groupby("kind").frame.describe()[["min", "50%", "max"]])
print("\njoint of the biggest jump:\n", jdf.groupby("kind").joint.value_counts().groupby(level=0).head(4))

In [ ]:
# 3b. Are the motion components just the speed setting? (uses D from check 3)
print("nearest other clip, mean abs diff (m): quantiles 0/10/50/90/100% =", np.quantile(nn, [0, .1, .5, .9, 1]).round(4))
n_comp, lab = connected_components(csr_matrix(D < 0.02), directed=False)
cm = pd.DataFrame([meta(c) for c in clips]).assign(motion=lab)
print(pd.crosstab(cm.motion, [cm.speed, cm.pos]))
for s in ("Speed_2", "Speed_3"):                  # largest difference INSIDE one speed setting
    i = np.where(cm.speed == s)[0]
    print(s, "max mean-abs-diff between any two clips (m):", D[np.ix_(i, i)].max().round(4))

In [ ]:
# 4b. Robust speed: slope of a line fit to MidHip (metres, camera frame), last frame dropped
rows = []
for c in clips:
    a, names = load_clean(c, "3D_cam")
    p = a[:, names.index("MidHip")]
    t = np.arange(len(p)) / 60
    v = np.polyfit(t, p, 1)[0]                     # (3,) metres per second
    rows.append(dict(**meta(c), clip=c, speed_ms=np.linalg.norm(v), vx=v[0], vy=v[1], vz=v[2]))
sp = pd.DataFrame(rows)
print(sp.groupby(["speed", "view"]).speed_ms.describe()[["mean", "std", "min", "max"]].round(3))
print("\nspread of speed across the 5 views of one group (m/s):")
print(sp.groupby("group").speed_ms.agg(lambda s: s.max() - s.min()).describe().round(3))

In [ ]:
# 5. Peek the SMPL motions zip (~60 MB)
smpl_zip = f"{LOCAL_DIR}/smpl_swimming_motions.zip"
if not os.path.exists(smpl_zip):
    subprocess.run(["wget", "-q", "-O", smpl_zip,
                    f"https://zenodo.org/records/{ZENODO_RECORD}/files/smpl_swimming_motions.zip?download=1"], check=True)
sz = zipfile.ZipFile(smpl_zip)
smpl_files = [i.filename for i in sz.infolist() if not i.is_dir()]
print(len(smpl_files), "files", dict(Counter(os.path.splitext(n)[1] for n in smpl_files)))
print(Counter("/".join(n.split("/")[:-1]) for n in smpl_files))
print(smpl_files[:10])
sz.extractall(f"{LOCAL_DIR}/smpl")
peek(os.path.join(LOCAL_DIR, "smpl", smpl_files[0]))

## Part 4 · SMPL motions and the 3D_cam jumps

Part 3 findings: all 2,880 Backstroke video clips replay **one** motion (Speed_3 clips are identical, Speed_2 differs
only by one outlier clip). So the 240 SMPL motions are the only source of motion variety. `3D_cam` has jumps of
~2 m in one frame (70× a normal move) at a different frame in each clip, which breaks the speed fit.

- 6. What arrays are in each SMPL `.npz` (poses, root translation, frame rate)? Speed per motion from root translation.
- 7. What do the `3D_cam` jumps look like: does the whole body move (camera cut) or one joint?

In [ ]:
# 6. SMPL motions: keys, shapes, and speed from the root translation
SMPL_DIR = f"{LOCAL_DIR}/smpl"
d = np.load(os.path.join(SMPL_DIR, smpl_files[0]))
for k in d.files:
    v = d[k]
    print(f"{k:16s} {v.shape} {v.dtype}", v if v.size < 12 else f"min {v.min():.3f} max {v.max():.3f}")

rows = []
for f in smpl_files:
    d = np.load(os.path.join(SMPL_DIR, f))
    fps = float(d["mocap_framerate"]) if "mocap_framerate" in d.files else np.nan
    tr = d["trans"] if "trans" in d.files else None
    row = dict(stroke=f.split("/")[0], file=f, frames=len(d["poses"]), fps=fps)
    if tr is not None:
        step = np.linalg.norm(np.diff(tr, axis=0), axis=1)
        row.update(net_ms=np.linalg.norm(tr[-1] - tr[0]) / ((len(tr) - 1) / fps),
                   med_step_ms=np.median(step) * fps, max_step_m=step.max(),
                   main_axis=int(np.argmax(np.abs(tr[-1] - tr[0]))))
    rows.append(row)
mo = pd.DataFrame(rows)
print(mo.groupby("stroke").describe().T.round(3).to_string())

In [ ]:
# 7. 3D_cam jumps: plot MidHip (camera frame) for the 5 views of one group, and how many joints jump together
g = meta(clips[0])["group"]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for c in [c for c in clips if meta(c)["group"] == g]:
    a, names = load_clean(c, "3D_cam")
    p = a[:, names.index("MidHip")]
    for k in range(3):
        ax[k].plot(p[:, k], label=meta(c)["view"], lw=1)
    step = np.linalg.norm(np.diff(a, axis=0), axis=2)          # (T-1, J)
    big = np.where(step.max(1) > 0.3)[0]
    print(f"{meta(c)['view']:17s} frames with a >0.3 m step: {big.tolist()} | joints moving >0.3 m in those frames:",
          [int((step[f] > 0.3).sum()) for f in big])
for k, n in enumerate("xyz"):
    ax[k].set_title(f"MidHip {n} (m, camera frame)"); ax[k].set_xlabel("frame")
ax[0].legend(fontsize=7); plt.show()

## Part 5 · Are the 60 SMPL motions per stroke really different?

Part 4 findings:
- SMPL `trans` is the same straight line in all 240 motions: exactly 2.5 m/s along y, 360 frames at 30 fps (12 s).
  The forward speed is added by the generator, not captured, so SwimXYZ holds **no real speed information**.
- The `3D_cam` "jumps" are the swimmer **turning around**: MidHip moves linearly (~2 m/s) then reverses direction
  at one frame per clip (Aerial frame 64 matches the 2D flip at ~65). Video translation is synthetic too.

So the only real signal is the body motion in `poses`. This cell checks it without the SMPL body model:
`poses` is axis-angle per joint (52 joints × 3, SMPL-H); columns 3:66 are the 21 body joints, root excluded.
- Stroke period per motion (autocorrelation of the body pose).
- Distance between motions after the best time shift, and how many near-duplicates there are.

In [ ]:
# 8. Variety inside the 60 SMPL motions of each stroke
from scipy.signal import find_peaks

BODY = slice(3, 66)
def load_pose(f):
    return np.load(os.path.join(SMPL_DIR, f))["poses"][:, BODY].astype(np.float32)

def period_s(x, fps=30):
    x = x - x.mean(0)
    ac = np.array([np.mean(np.sum(x[:-k] * x[k:], 1)) for k in range(1, 180)])
    pk, _ = find_peaks(ac, prominence=0.05 * ac.max())
    return (pk[0] + 1) / fps if len(pk) else np.nan

def shift_dist(a, b, max_lag=90, n=240):
    # mean abs pose difference (rad) at the best time shift of b against a
    return min(np.abs(a[:n] - b[k:k + n]).mean() for k in range(max_lag + 1))

betas = np.stack([np.load(os.path.join(SMPL_DIR, f))["betas"] for f in smpl_files])
genders = Counter(str(np.load(os.path.join(SMPL_DIR, f))["gender"]) for f in smpl_files)
print("betas std across all 240 motions:", betas.std(0).max().round(4), "| genders:", dict(genders))

poses = {f: load_pose(f) for f in smpl_files}
for stroke in sorted({f.split("/")[0] for f in smpl_files}):
    fs = sorted([f for f in smpl_files if f.startswith(stroke + "/")],
                key=lambda f: int(f.split("_")[-2]))
    per = np.array([period_s(poses[f]) for f in fs])
    Dm = np.array([[shift_dist(poses[a], poses[b]) if a != b else np.nan for b in fs] for a in fs])
    nn_m = np.nanmin(Dm, 1)
    print(f"\n{stroke}: period (s) quantiles 0/25/50/75/100% = {np.nanquantile(per, [0, .25, .5, .75, 1]).round(2)}")
    print(f"  nearest other motion (rad): quantiles = {np.quantile(nn_m, [0, .25, .5, .75, 1]).round(3)}"
          f" | median pair = {np.nanmedian(Dm):.3f} | pairs < 0.02 rad: {int(np.nansum(Dm < 0.02) // 2)}")
    print("  first 12 periods:", per[:12].round(2))

# scale reference: distance between different strokes
s = sorted({f.split("/")[0] for f in smpl_files})
a, b = [next(f for f in smpl_files if f.startswith(x + "/")) for x in s[:2]]
print(f"\nreference: {a} vs {b} = {shift_dist(poses[a], poses[b]):.3f} rad")

## What to send back

Copy the printed output of the **structure** and **peek** cells. From it we decide:

1. How a video is identified (file name pattern → stroke, view, speed, motion id).
2. Where 2D joints (pixels?) and 3D joints (global vs pelvis frame) live.
3. Whether the global 3D pelvis track exists → ground-truth **speed** and **distance per stroke**.

Then notebook 02 parses everything into one compact `.npz` on Drive.